# P07 — Línea base: TF-IDF + SVM
## (a) Paso del protocolo y objetivo
**Paso P07**: entrenar un clasificador clásico (TF-IDF de palabras 1–2-gramas + SVM lineal, `C ∈ {0,1; 1; 10}`, semilla 42) como **línea base**. Sirve para saber si Rasa/DIET (P08) aporta algo: el **OE2** pide una arquitectura justificada, y una línea base simple es la comparación honesta. **`C` se elige en validación (o validación cruzada), nunca en el test.**
**Origen:** entrenamiento de demostración con el corpus **Sintético**; resultados guardados sobre lenguaje **Real** (lotes 1 y 2) solo como cifras.

## (b) Código del repositorio que lo implementa
`train_baseline_p07.py` (primer experimento, corpus v1 de 324 frases), `train_baseline.py` (P07/P10/P11 con la grilla de `configs/baseline_config.json`) y `entrenar_svm_lote2.py` (el SVM del lote 2, con las **mismas 943 frases** que DIET). Entradas: corpus particionado y `configs/baseline_config.json`. Salidas: `logs/<experimento>/` con métricas y predicciones; `models/svm/LOTE2-SVM.joblib` (no se versiona).

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("07"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,demo_vivo.py,"Demostración en vivo (P07, P08)",Piezas en Python de la demostración en vivo (las orquesta scripts/demo_vivo.ps1; se pueden correr sueltas).,"Corpus sintético, configs/","models/demo_vivo/ (SVM y DIET de demostración), huellas de archivos protegidos, tabla de pruebas",python scripts/demo_vivo.py snapshot guardar|comparar huellas de los archivos protegidos (antes y después de la demo),No,Documentado (se ejecuta con demo_vivo.ps1; requiere Rasa)
1,entrenar_svm_lote2.py,Lote 2 (baseline),Lote 2 — entrena el baseline SVM (TF-IDF + SVC lineal) con el MISMO conjunto de entrenamiento que DIET (corpus/v3_lote2/entrenamiento_lote2.csv: 707 sintéticas + 51 sintéticas nuevas + 185 reales = 943) y lo gu,entrenamiento_lote2.csv,models/svm/LOTE2-SVM.joblib,python scripts/entrenar_svm_lote2.py [--c 10],No,Documentado (el SVM del lote 2 ya está congelado)
2,train_baseline.py,"P07, P10, P11",P07 / P10 / P11 — Baseline de Machine Learning: TF-IDF + SVM (y regresión logística).,corpus + configs/baseline_config.json,logs/<experimento>/ con métricas y predicciones (TF-IDF + SVM y regresión logística),python scripts/train_baseline.py,No,"Lógica reproducida en pequeño (etapa 5, sintético)"
3,train_baseline_p07.py,P07 (versión inicial),"P07 del Protocolo V1.1: Experimento baseline (TF-IDF + SVM) sobre el Corpus MPSR-Bot real (324 utterances, 54 intenciones, 9 categorías). EJECUCIÓN REAL (no simulada) -- los resultados de este script son el des",Corpus v1 real de 324 frases,logs/EXP_BASELINE_SVM_S42_2026/,python scripts/train_baseline_p07.py --help,No,Documentado (histórico)


## (c) Celdas de código
### 1. Parámetros fijados antes de entrenar y resultados guardados de la línea base
(La primera celda muestra también los parámetros de DIET, que se tratan en el cuaderno 08.)

In [3]:
import yaml
cfg = yaml.safe_load((BASE / "configs/rasa_config_lote2.yml").read_text(encoding="utf-8"))
bcfg = leer_json("configs/baseline_config.json")
diet = next(c for c in cfg["pipeline"] if c["name"] == "DIETClassifier")
fb = next(c for c in cfg["pipeline"] if c["name"] == "FallbackClassifier")
rotulo(ORIGEN_SINTETICO, "configuración (código, no datos)")
display(pd.DataFrame([
    {"modelo": "DIET (congelado)", "parámetro": k, "valor": diet[k]} for k in ("epochs", "batch_size", "embedding_dimension", "random_seed", "learning_rate")] + [
    {"modelo": "FallbackClassifier", "parámetro": "threshold", "valor": fb["threshold"]}, {"modelo": "FallbackClassifier", "parámetro": "ambiguity_threshold", "valor": fb["ambiguity_threshold"]}] + [
    {"modelo": "SVM baseline", "parámetro": "vectorizador", "valor": f"TF-IDF max_features={bcfg['vectorizer']['max_features']}, ngram_range={tuple(bcfg['vectorizer']['ngram_range'])}"},
    {"modelo": "SVM baseline", "parámetro": "clasificador", "valor": f"SVC kernel={bcfg['classifier']['kernel']}, C ∈ {bcfg['classifier']['C_grid']}"},
    {"modelo": "SVM baseline", "parámetro": "criterio de selección", "valor": bcfg["classifier"]["selection_criterion"]}]))
comparar("P07/P08", "DIET epochs/batch/dim/semilla", f"{diet['epochs']}/{diet['batch_size']}/{diet['embedding_dimension']}/{diet['random_seed']}", "100/64/20/42", "configs/rasa_config_lote2.yml", tipo="txt")
comparar("P07/P08", "umbral t y ambigüedad", f"{fb['threshold']}/{fb['ambiguity_threshold']}", "0.5/0.1", "configs/rasa_config_lote2.yml", tipo="txt")

▌ORIGEN: Sintético — configuración (código, no datos)


,modelo,parámetro,valor
0,DIET (congelado),epochs,100
1,DIET (congelado),batch_size,64
2,DIET (congelado),embedding_dimension,20
3,DIET (congelado),random_seed,42
4,DIET (congelado),learning_rate,0.001
5,FallbackClassifier,threshold,0.5
6,FallbackClassifier,ambiguity_threshold,0.1
7,SVM baseline,vectorizador,"TF-IDF max_features=5000, ngram_range=(1, 2)"
8,SVM baseline,clasificador,"SVC kernel=linear, C ∈ [0.1, 1, 10]"
9,SVM baseline,criterio de selección,mejor F1 macro en validación


In [4]:
# ----- Línea base guardada (corpus sintético, partición v3 sintética) -----
rotulo(ORIGEN_SINTETICO, "resultados guardados de la línea base (logs/baseline_*.csv)")
bv = pd.read_csv(BASE / "logs/baseline_validation.csv")
bt = pd.read_csv(BASE / "logs/baseline_test.csv")
print("Validación:")
display(bv[["experiment_id", "model", "C", "seed", "accuracy", "f1_macro"]].round(4))
print("(El test sintético se muestra solo como registro histórico del protocolo; la decisión de C se tomó en validación.)")
display(bt[["experiment_id", "model", "C", "seed", "accuracy", "f1_macro"]].round(4))

▌ORIGEN: Sintético — resultados guardados de la línea base (logs/baseline_*.csv)
Validación:


,experiment_id,model,C,seed,accuracy,f1_macro
0,BASE-SVM-C0.1-s42,svm,0.1,42,0.0000,0.0000
1,BASE-SVM-C1-s42,svm,1.0,42,0.7778,0.6532
2,BASE-SVM-C10-s42,svm,10.0,42,0.7901,0.6578
3,BASE-LOGREG-C1.0-s42,logreg,1.0,42,0.7407,0.6632


,experiment_id,model,C,seed,accuracy,f1_macro
0,BASE-SVM-C10-s10,svm,10.0,10,0.7901,0.6480
1,BASE-SVM-C10-s20,svm,10.0,20,0.7901,0.6480
2,BASE-SVM-C10-s30,svm,10.0,30,0.7901,0.6480
3,BASE-SVM-C10-s40,svm,10.0,40,0.7901,0.6480
4,BASE-SVM-C10-s50,svm,10.0,50,0.7901,0.6480
5,BASE-LOGREG-C1.0-s10,logreg,1.0,10,0.7037,0.5786
6,BASE-LOGREG-C1.0-s20,logreg,1.0,20,0.7037,0.5786
7,BASE-LOGREG-C1.0-s30,logreg,1.0,30,0.7037,0.5786
8,BASE-LOGREG-C1.0-s40,logreg,1.0,40,0.7037,0.5786
9,BASE-LOGREG-C1.0-s50,logreg,1.0,50,0.7037,0.5786


### 2. Demostración de entrenamiento (Sintético): elegir `C` en validación y validación cruzada agrupada
La celda siguiente necesita el corpus sintético cargado.

In [5]:
cm = pd.read_csv(BASE / "corpus/corpus_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8")

In [6]:
# ----- Demostración de entrenamiento (SINTÉTICO) en carpeta aparte -----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold

DEMO_DIR = BASE / "demo_entrenamiento"
assert "modelo_congelado" not in str(DEMO_DIR) and "models" not in DEMO_DIR.parts, "la demostración nunca escribe junto al modelo congelado"
DEMO_DIR.mkdir(exist_ok=True)
(DEMO_DIR / "LEEME_demo.txt").write_text("DEMOSTRACIÓN (Sintético): no sustituye al modelo congelado LOTE2-FINAL v1. Entrenado en el cuaderno con el corpus sintético; evaluado solo en validación.\n", encoding="utf-8")

try:
    sys.path.insert(0, str(BASE / "scripts"))
    import common
    jerga = common.load_jerga()
    normalizar = lambda t: common.normalize(t, jerga)
    print("Normalización de jerga local: scripts/common.py (configs/jerga_local.csv)")
except Exception as e:  # sin jerga: se usa el texto en minúsculas
    normalizar = lambda t: t.lower()
    print("Aviso: no se pudo cargar common.py, se usa minúsculas:", str(e)[:80])

tr, va = cm[cm["split"] == "train"], cm[cm["split"] == "validation"]
vec = TfidfVectorizer(max_features=bcfg["vectorizer"]["max_features"], ngram_range=tuple(bcfg["vectorizer"]["ngram_range"]))
Xtr = vec.fit_transform(tr["text"].map(normalizar)); Xva = vec.transform(va["text"].map(normalizar))
resultados = []
for C in bcfg["classifier"]["C_grid"]:
    clf = SVC(kernel="linear", C=C, random_state=42).fit(Xtr, tr["intent"])
    resultados.append({"C": C, "F1 macro en validación": round(f1_score(va["intent"], clf.predict(Xva), average="macro", zero_division=0), 4)})
rv = pd.DataFrame(resultados)
mejorC = float(rv.sort_values("F1 macro en validación", ascending=False).iloc[0]["C"])
rotulo(ORIGEN_SINTETICO, "DEMOSTRACIÓN — no sustituye al modelo congelado; evaluado SOLO en validación")
display(rv)
print("C elegido en validación:", mejorC, "(el test no se usa para decidir)")

# validación cruzada agrupada por base_phrase_id dentro del entrenamiento (sin fuga por paráfrasis)
gkf = GroupKFold(n_splits=5)
f1s = []
for a, b in gkf.split(tr, tr["intent"], tr["base_phrase_id"]):
    v = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    Xa = v.fit_transform(tr.iloc[a]["text"].map(normalizar)); Xb = v.transform(tr.iloc[b]["text"].map(normalizar))
    m = SVC(kernel="linear", C=mejorC, random_state=42).fit(Xa, tr.iloc[a]["intent"])
    f1s.append(f1_score(tr.iloc[b]["intent"], m.predict(Xb), average="macro", zero_division=0))
print("Validación cruzada agrupada (5 folds, por base_phrase_id) — F1 macro por fold:", [round(x, 3) for x in f1s], "| media", round(float(np.mean(f1s)), 3))

Normalización de jerga local: scripts/common.py (configs/jerga_local.csv)
▌ORIGEN: Sintético — DEMOSTRACIÓN — no sustituye al modelo congelado; evaluado SOLO en validación


,C,F1 macro en validación
0,0.1,0.0000
1,1.0,0.6532
2,10.0,0.6578


### 3. La línea base sobre lenguaje real (Real, cifras guardadas)
En el **lote 1** el SVM se eligió en validación (`C = 10`); en el **lote 2** el SVM se entrenó con el mismo conjunto de 943 frases que DIET y se evaluó **una sola vez** junto con DIET (solo informativo: G3 se mide con DIET).

In [7]:
sel = leer_json("logs/v3_real/seleccion_final.json")
res = leer_json("logs/avance/eval_lote2_resumen.json")
svm2, rasa2 = res["metodos"]["svm"], res["metodos"]["rasa"]
rotulo(ORIGEN_REAL, "resultados guardados (no recalculados aquí)")
display(pd.DataFrame([
    {"medición": "Lote 1 — validación (selección de C)", "modelo": "SVM C = 10", "F1 macro": sel["svm"]["f1_macro_validacion"], "IC95 % inferior": sel["svm"]["validacion_ic95"]["f1_macro"][1], "IC95 % superior": sel["svm"]["validacion_ic95"]["f1_macro"][2]},
    {"medición": "Lote 1 — validación (selección de DIET)", "modelo": "DIET 100/64/20", "F1 macro": sel["rasa"]["f1_macro_validacion"], "IC95 % inferior": sel["rasa"]["validacion_ic95"]["f1_macro"][1], "IC95 % superior": sel["rasa"]["validacion_ic95"]["f1_macro"][2]},
    {"medición": "Lote 2 — test único (informativo)", "modelo": "SVM", "F1 macro": svm2["f1_macro"][0], "IC95 % inferior": svm2["f1_macro"][1], "IC95 % superior": svm2["f1_macro"][2]},
    {"medición": "Lote 2 — test único (G3)", "modelo": "DIET congelado", "F1 macro": rasa2["f1_macro"][0], "IC95 % inferior": rasa2["f1_macro"][1], "IC95 % superior": rasa2["f1_macro"][2]},
]).round(4))
cr = svm2["comparacion_con_rasa"]
print(f"Lote 2: solo el SVM acierta {cr['solo_svm_acierta']} frases, solo DIET acierta {cr['solo_rasa_acierta']}; McNemar exacto p = {cr['p_mcnemar_exacto']:.3f} (no hay evidencia de que DIET sea mejor que el SVM).")
comparar("P07", "F1 macro del SVM en el test del lote 2", svm2["f1_macro"][0], 0.8925, "Nota v11 / eval_lote2_resumen.json", tol=5e-4)
comparar("P07", "McNemar SVM vs DIET (p exacto)", cr["p_mcnemar_exacto"], 0.541, "eval_lote2_resumen.json", tol=5e-4)
previo = leer_json("logs/v3_real/lote2_congelado_previo_svm.json")
print("SVM del lote 2 congelado antes de abrir el lote 2 — sha256 del modelo:", previo["sha256"]["modelo"][:12], "… | del entrenamiento:", previo["sha256"]["corpus"][:12], "…")
comparar("P07", "el SVM se entrenó con el mismo entrenamiento de 943 frases que DIET", previo["sha256"]["corpus"], leer_json("logs/v3_real/modelo_congelado.json")["sha256"]["corpus"], "congelamientos previos", tipo="txt")

cvp = (BASE / "logs/avance/cv_participantes_resumen.md").read_text(encoding="utf-8").splitlines()
print("\nValidación cruzada por participantes del lote 1 (análisis de sensibilidad, no cuenta para G3):")
for l in cvp[:8]:
    if l.startswith(("- F1 macro global", "# ", "185 frases")):
        print(" ", l[:330])

▌ORIGEN: Real — resultados guardados (no recalculados aquí)


,medición,modelo,F1 macro,IC95 % inferior,IC95 % superior
0,Lote 1 — validación (selección de C),SVM C = 10,0.6304,0.5021,0.7058
1,Lote 1 — validación (selección de DIET),DIET 100/64/20,0.6809,0.5414,0.7370
2,Lote 2 — test único (informativo),SVM,0.8925,0.8417,0.9171
3,Lote 2 — test único (G3),DIET congelado,0.9072,0.8626,0.9290


In [8]:
cierre("P07 Baseline SVM")

P07 Baseline SVM — recalculado frente a reportado: 5 cifras · coinciden: 5 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P07/P08,DIET epochs/batch/dim/semilla,100/64/20/42,100/64/20/42,configs/rasa_config_lote2.yml,Sí
1,P07/P08,umbral t y ambigüedad,0.5/0.1,0.5/0.1,configs/rasa_config_lote2.yml,Sí
2,P07,F1 macro del SVM en el test del lote 2,0.892452,0.8925,Nota v11 / eval_lote2_resumen.json,Sí
3,P07,McNemar SVM vs DIET (p exacto),0.541256,0.541,eval_lote2_resumen.json,Sí
4,P07,el SVM se entrenó con el mismo entrenamiento de 943 frases que DIET,c035252d63aa89e11295279b363c98d66b9f2e608008b82390b15ce40a6a4adc,c035252d63aa89e11295279b363c98d66b9f2e608008b82390b15ce40a6a4adc,congelamientos previos,Sí


## (d) Cómo leer el resultado
- La **demostración** muestra el procedimiento: se prueba cada `C` en validación, se elige el de mayor F1 macro y se comprueba con validación cruzada **agrupada** por `base_phrase_id`. El test sintético no participa en la decisión.
- Sobre **lenguaje real**, el SVM queda cerca de DIET: en el lote 2 el F1 macro es 0,892 (SVM) frente a 0,907 (DIET), con p = 0,541 en McNemar. Eso significa que la ventaja de DIET **no está demostrada**; la elección de Rasa se apoya también en la arquitectura del proyecto (diálogo, umbral con `FallbackClassifier`), no solo en el F1.

## Limitaciones
- Con 4–5 frases por intención en el test, diferencias de 0,015 en F1 caen dentro del ruido.
- Los resultados de la demostración (corpus sintético) **no son comparables** con los del lenguaje real.
- El modelo `.joblib` del SVM no se carga aquí: se guardó con scikit-learn 1.1.3 y Colab trae otra versión.